# 01 · First look at the data

Run `make fetch && make clean-data` first (or `uv run sft fetch` / `uv run sft clean`).

All events are *apparent* events derived from AIS by Global Fishing Watch.

In [ ]:
import pandas as pd
import plotly.express as px
from sft.config import PROCESSED
from sft.fetch import load_vessels

vessels = load_vessels()
ev = pd.read_parquet(PROCESSED / 'events.parquet')
ids = pd.read_parquet(PROCESSED / 'identities.parquet')
vessels

## Event counts per vessel

In [ ]:
ev.groupby(['imo', 'type']).size().unstack(fill_value=0)

## Identity history: name, flag and MMSI changes

Each change is something to explain in the write-up.

In [ ]:
from sft.ownership import identity_changes
identity_changes(ids)

## Timeline of events

In [ ]:
names = vessels.set_index('imo')['name']
t = ev.assign(vessel=ev['imo'].map(names))
px.timeline(t, x_start='start', x_end='end', y='vessel', color='type', height=400)

## AIS gap length: sanctioned vs control

This distribution is what justifies the `long_gap` threshold in METHODOLOGY.md.

In [ ]:
groups = vessels.set_index('imo')['group']
gaps = ev[ev['type'] == 'gap'].assign(group=lambda d: d['imo'].map(groups))
px.histogram(gaps, x='duration_h', color='group', barmode='overlay', nbins=60, log_y=True)